In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import vitaldb as v

import mlflow

c:\Users\mikol\anaconda3\envs\master_thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

path = '../../data/processed/eeg_sample.csv'
df = pd.read_csv(path)
X = df[['delta', 'theta', 'alpha', 'beta']].to_numpy()
y = df['bis'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# n_jobs=-1 uses all available CPU cores to speed up training
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

print("Training the Random Forest model (this might take a moment depending on data size)...")
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Model Evaluation ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.2f}")

# Optional: Look at feature importance
print("\n--- Feature Importance ---")
for feature, importance in zip(['delta', 'theta', 'alpha', 'beta'], rf_model.feature_importances_):
    print(f"{feature}: {importance:.4f}")

Training the Random Forest model (this might take a moment depending on data size)...

--- Model Evaluation ---
Mean Squared Error (MSE): 174.87
Mean Absolute Error (MAE): 8.44
R-squared (R2): 0.43

--- Feature Importance ---
delta: 0.1618
theta: 0.1636
alpha: 0.1936
beta: 0.4810


With MLflow

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import mlflow
import mlflow.sklearn

# 1. Setup MLflow
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://localhost:5000")
# Specify the experiment you just created.
mlflow.set_experiment("Random Forest")

# 2. Load Data
path = '../../data/processed/eeg_sample.csv'
df = pd.read_csv(path)
X = df[['delta', 'theta', 'alpha', 'beta']].to_numpy()
y = df['bis'].to_numpy()

# Define your hyperparameters in a dictionary upfront so they are easy to log
params = {
    "test_size": 0.2,
    "random_state": 42,
    "n_estimators": 100,
    "shuffle": False  # Crucial to prevent time-series data leakage!
}

# 3. Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=params["test_size"], 
    random_state=params["random_state"], 
    shuffle=params["shuffle"]
)

# ---------------------------------------------------------
# 4. Start the MLflow Run
# Everything indented under this 'with' block gets logged together
# ---------------------------------------------------------
with mlflow.start_run(run_name="EEG_BIS_Baseline"):
    
    # Log all your hyperparameters at once
    mlflow.log_params(params)
    
    # Initialize and Train the Model
    rf_model = RandomForestRegressor(
        n_estimators=params["n_estimators"], 
        random_state=params["random_state"], 
        n_jobs=-1
    )
    
    print("Training the Random Forest model...")
    rf_model.fit(X_train, y_train)

    # Make Predictions
    y_pred = rf_model.predict(X_test)

    # Evaluate the Performance
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n--- Model Evaluation ---")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"R-squared (R2): {r2:.2f}")

    # Log your evaluation metrics to MLflow
    mlflow.log_metrics({
        "mse": mse,
        "mae": mae,
        "r2": r2
    })

    # Log feature importances as metrics so you can track them over time
    feature_names = ['delta', 'theta', 'alpha', 'beta']
    importances = {f"importance_{feat}": imp for feat, imp in zip(feature_names, rf_model.feature_importances_)}
    mlflow.log_metrics(importances)

    # Log the actual model file (creates a serialized version of your model)
    # The "random_forest_model" is the folder name MLflow will save it under
    mlflow.sklearn.log_model(rf_model, "random_forest_model")
    
    print("\n✅ Run successfully logged to MLflow!")

Training the Random Forest model...


2026/04/22 16:04:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



--- Model Evaluation ---
Mean Squared Error (MSE): 174.87
Mean Absolute Error (MAE): 8.44
R-squared (R2): 0.43


2026/04/22 16:04:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run successfully logged to MLflow!
🏃 View run EEG_BIS_Baseline at: http://localhost:5000/#/experiments/1/runs/fbbc81ba4f7e4bfbb19a7c3062a3a1d2
🧪 View experiment at: http://localhost:5000/#/experiments/1
